# 扩散模型采样策略详解

本教程深入讲解扩散模型的各种采样策略：
- DDPM 标准采样
- DDIM 加速采样
- DPM-Solver 高阶求解器
- Classifier-Free Guidance

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
import math
from typing import Optional, List, Tuple

from diffusion import DDPM, DiffusionConfig, NoiseScheduler, create_diffusion_model

## 1. DDPM 标准采样

### 数学原理

$$p_\theta(x_{t-1}|x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \sigma_t^2 I)$$

$$\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t, t)\right)$$

In [ ]:
class DDPMSampler:
    """DDPM 标准采样器"""
    
    def __init__(self, model: DDPM):
        self.model = model
        self.scheduler = model.scheduler
    
    @torch.no_grad()
    def sample(self, batch_size: int, device: torch.device) -> torch.Tensor:
        """DDPM 采样 - 需要 T 步"""
        config = self.model.config
        
        # 从纯噪声开始
        x = torch.randn(batch_size, config.in_channels, config.image_size, config.image_size, device=device)
        
        for t in reversed(range(config.num_timesteps)):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
            
            # 预测噪声
            noise_pred = self.model.unet(x, t_batch)
            
            # 计算均值
            alpha_t = self.scheduler.alphas[t].to(device)
            alpha_bar_t = self.scheduler.alphas_cumprod[t].to(device)
            beta_t = self.scheduler.betas[t].to(device)
            
            mean = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * noise_pred)
            
            # 添加噪声 (t > 0)
            if t > 0:
                noise = torch.randn_like(x)
                sigma = torch.sqrt(beta_t)
                x = mean + sigma * noise
            else:
                x = mean
        
        return x

# 测试
model = create_diffusion_model("small")
sampler = DDPMSampler(model)
print(f"DDPM sampler created, requires {model.config.num_timesteps} steps")

## 2. DDIM 加速采样

DDIM 使用确定性采样，可以大幅减少步数。

In [ ]:
class DDIMSampler:
    """DDIM 加速采样器"""
    
    def __init__(self, model: DDPM, num_inference_steps: int = 50):
        self.model = model
        self.scheduler = model.scheduler
        self.num_inference_steps = num_inference_steps
        self.timesteps = self._get_timesteps()
    
    def _get_timesteps(self) -> torch.Tensor:
        step_ratio = self.scheduler.num_timesteps // self.num_inference_steps
        timesteps = torch.arange(0, self.num_inference_steps) * step_ratio
        return timesteps.flip(0).long()
    
    @torch.no_grad()
    def sample(self, batch_size: int, device: torch.device, eta: float = 0.0) -> torch.Tensor:
        """
        DDIM 采样
        
        Args:
            eta: 随机性参数 (0=确定性, 1=DDPM)
        """
        config = self.model.config
        x = torch.randn(batch_size, config.in_channels, config.image_size, config.image_size, device=device)
        
        timesteps = self.timesteps.to(device)
        
        for i, t in enumerate(timesteps):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
            
            # 预测噪声
            noise_pred = self.model.unet(x, t_batch)
            
            # 获取 alpha 值
            alpha_t = self.scheduler.alphas_cumprod[t].to(device)
            t_prev = timesteps[i + 1] if i < len(timesteps) - 1 else torch.tensor(0)
            alpha_t_prev = self.scheduler.alphas_cumprod[t_prev].to(device) if t_prev >= 0 else torch.tensor(1.0, device=device)
            
            # 预测 x_0
            x0_pred = (x - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            x0_pred = torch.clamp(x0_pred, -1, 1)
            
            # 计算方差
            sigma_t = eta * torch.sqrt((1 - alpha_t_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_t_prev)
            
            # DDIM 更新
            dir_xt = torch.sqrt(1 - alpha_t_prev - sigma_t ** 2) * noise_pred
            x = torch.sqrt(alpha_t_prev) * x0_pred + dir_xt
            
            if eta > 0 and i < len(timesteps) - 1:
                x = x + sigma_t * torch.randn_like(x)
        
        return x

# 测试
ddim_sampler = DDIMSampler(model, num_inference_steps=50)
print(f"DDIM sampler: {ddim_sampler.num_inference_steps} steps (vs {model.config.num_timesteps} for DDPM)")

## 3. DPM-Solver 高阶求解器

In [ ]:
class DPMSolverSampler:
    """
    DPM-Solver 采样器 (简化版)
    
    使用高阶 ODE 求解器加速采样，通常只需 20-25 步。
    """
    
    def __init__(self, model: DDPM, num_inference_steps: int = 20, order: int = 2):
        self.model = model
        self.scheduler = model.scheduler
        self.num_inference_steps = num_inference_steps
        self.order = order
        
        # 计算 lambda (log-SNR)
        self.lambdas = self._compute_lambdas()
        self.timesteps = self._get_timesteps()
    
    def _compute_lambdas(self) -> torch.Tensor:
        """计算 log-SNR"""
        alpha_bar = self.scheduler.alphas_cumprod
        return 0.5 * torch.log(alpha_bar / (1 - alpha_bar))
    
    def _get_timesteps(self) -> torch.Tensor:
        step_ratio = self.scheduler.num_timesteps // self.num_inference_steps
        return torch.arange(0, self.num_inference_steps) * step_ratio
    
    @torch.no_grad()
    def sample(self, batch_size: int, device: torch.device) -> torch.Tensor:
        """DPM-Solver 采样"""
        config = self.model.config
        x = torch.randn(batch_size, config.in_channels, config.image_size, config.image_size, device=device)
        
        timesteps = self.timesteps.flip(0).to(device)
        model_outputs = []  # 存储历史输出用于高阶更新
        
        for i, t in enumerate(timesteps):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
            
            # 预测噪声
            noise_pred = self.model.unet(x, t_batch)
            model_outputs.append(noise_pred)
            
            if i < len(timesteps) - 1:
                t_next = timesteps[i + 1]
                
                # 一阶更新 (简化)
                alpha_t = self.scheduler.alphas_cumprod[t].to(device)
                alpha_next = self.scheduler.alphas_cumprod[t_next].to(device)
                
                x0_pred = (x - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
                x = torch.sqrt(alpha_next) * x0_pred + torch.sqrt(1 - alpha_next) * noise_pred
        
        return x

dpm_sampler = DPMSolverSampler(model, num_inference_steps=20)
print(f"DPM-Solver: {dpm_sampler.num_inference_steps} steps")

## 4. Classifier-Free Guidance

In [ ]:
class CFGSampler:
    """
    Classifier-Free Guidance 采样器
    
    公式: ε̃ = ε_uncond + w * (ε_cond - ε_uncond)
    """
    
    def __init__(self, model: DDPM, num_inference_steps: int = 50):
        self.model = model
        self.scheduler = model.scheduler
        self.num_inference_steps = num_inference_steps
    
    @torch.no_grad()
    def sample(self, batch_size: int, device: torch.device, 
               class_labels: torch.Tensor, guidance_scale: float = 7.5) -> torch.Tensor:
        """
        CFG 采样
        
        Args:
            class_labels: 条件类别标签
            guidance_scale: 引导强度 (w)
        """
        config = self.model.config
        x = torch.randn(batch_size, config.in_channels, config.image_size, config.image_size, device=device)
        
        step_ratio = config.num_timesteps // self.num_inference_steps
        timesteps = torch.arange(0, self.num_inference_steps) * step_ratio
        timesteps = timesteps.flip(0).to(device)
        
        for i, t in enumerate(timesteps):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
            
            # 条件预测
            noise_cond = self.model.unet(x, t_batch, class_labels)
            
            # 无条件预测 (使用 null class)
            null_labels = torch.zeros_like(class_labels)
            noise_uncond = self.model.unet(x, t_batch, null_labels)
            
            # CFG 组合
            noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
            
            # DDIM 更新
            alpha_t = self.scheduler.alphas_cumprod[t].to(device)
            t_prev = timesteps[i + 1] if i < len(timesteps) - 1 else torch.tensor(0)
            alpha_prev = self.scheduler.alphas_cumprod[t_prev].to(device) if t_prev >= 0 else torch.tensor(1.0, device=device)
            
            x0_pred = (x - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            x0_pred = torch.clamp(x0_pred, -1, 1)
            
            dir_xt = torch.sqrt(1 - alpha_prev) * noise_pred
            x = torch.sqrt(alpha_prev) * x0_pred + dir_xt
        
        return x

print("CFG Sampler: guidance_scale 控制条件强度")
print("  - w=1.0: 标准条件生成")
print("  - w=7.5: 常用设置，增强条件一致性")
print("  - w>10: 更强条件，可能过饱和")

## 5. 采样策略对比

In [ ]:
# 采样策略对比表
comparison = """
| 采样器 | 步数 | 质量 | 速度 | 特点 |
|--------|------|------|------|------|
| DDPM | 1000 | 最高 | 最慢 | 标准采样，理论最优 |
| DDIM | 50-100 | 高 | 快 | 确定性采样，可逆 |
| DPM-Solver | 20-25 | 高 | 很快 | 高阶 ODE 求解器 |
| UniPC | 10-20 | 高 | 极快 | 统一预测校正器 |
"""
print(comparison)

## 总结

本教程介绍了扩散模型的核心采样策略：

1. **DDPM**: 标准采样，1000步，质量最高
2. **DDIM**: 加速采样，50步，确定性
3. **DPM-Solver**: 高阶求解器，20步
4. **CFG**: 增强条件控制的引导策略